In [1]:
import json
import duckdb
import pandas as pd
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB CONNECTION
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RESULT_DIR = BASE_DIR / "results"
FIG_DIR = RESULT_DIR / "figures"
ANALYSIS_DIR = BASE_DIR / "analysis"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = RESULT_DIR / "step03_01_star_schema.duckdb"
assert DB_PATH.exists(), f"Warehouse database not found at {DB_PATH}"

con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

# Load Step 3.1 findings to verify baseline thresholds and row counts
with open(RESULT_DIR / "step03_01_STAR_SCHEMA_FINDINGS.json", "r") as f:
    step31_findings = json.load(f)

THRESH_2SD = float(step31_findings["fact_delay_summary"]["threshold_2sd_minutes"])
THRESH_3SD = float(step31_findings["fact_delay_summary"]["threshold_3sd_minutes"])
EXPECTED_TOTAL_ROWS = int(step31_findings["fact_delay_summary"]["total_rows"])

print("=== CONNECTED TO STAR SCHEMA WAREHOUSE ===")
print(f"Database Path            : {DB_PATH.as_posix()}")
print(f"Total Fact Rows          : {EXPECTED_TOTAL_ROWS:,}")
print(f"2 SD High-Risk Threshold : {THRESH_2SD:.6f} min")
print(f"3 SD Critical Threshold  : {THRESH_3SD:.6f} min")

=== CONNECTED TO STAR SCHEMA WAREHOUSE ===
Database Path            : D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step03_01_star_schema.duckdb
Total Fact Rows          : 36,614,824
2 SD High-Risk Threshold : 167.653054 min
3 SD Critical Threshold  : 233.679870 min


In [2]:
# ==========================================
# STEP 1: CREATE REUSABLE ANALYTICAL VIEWS
# ==========================================
print("\n=== STEP 1: CREATING REUSABLE ANALYTICAL VIEWS ===")

con.execute("""
-- 1. Base enriched analytical view with explicit indicator columns (0/1) for consistent denominators
CREATE OR REPLACE VIEW vw_olap_base AS
SELECT 
    *,
    CASE WHEN delay < 0 THEN 1 ELSE 0 END AS is_early,
    CASE WHEN delay = 0 THEN 1 ELSE 0 END AS is_exact_ontime,
    CASE WHEN delay <= 0 THEN 1 ELSE 0 END AS is_early_or_ontime,
    CASE WHEN delay > 0 AND delay <= 15 THEN 1 ELSE 0 END AS is_minor_delay_1_15m,
    CASE WHEN delay <= 15 THEN 1 ELSE 0 END AS is_on_time_15m,
    CASE WHEN delay > 15 THEN 1 ELSE 0 END AS is_delayed_gt_15m,
    CASE WHEN delay > 15 AND delay <= 60 THEN 1 ELSE 0 END AS is_moderate_delay_15_60m,
    CASE WHEN delay > 60 THEN 1 ELSE 0 END AS is_severe_delay_gt_60m,
    CASE WHEN severity = 'high_risk' THEN 1 ELSE 0 END AS is_high_risk_only,
    CASE WHEN severity = 'critical' THEN 1 ELSE 0 END AS is_critical_only,
    CASE WHEN severity IN ('high_risk', 'critical') THEN 1 ELSE 0 END AS is_bottleneck_gt_2sd
FROM olap_star_schema;

-- 2. Default OLAP View: All Observed Records (is_imputed = FALSE)
CREATE OR REPLACE VIEW vw_olap_observed AS
SELECT * FROM vw_olap_base
WHERE is_imputed = FALSE;

-- 3. Imputed-Only OLAP View: Synthetic Records (is_imputed = TRUE)
CREATE OR REPLACE VIEW vw_olap_imputed AS
SELECT * FROM vw_olap_base
WHERE is_imputed = TRUE;

-- 4. Observed Arrival Delay View: Intermediate & Destination Stops (station_no > 1)
CREATE OR REPLACE VIEW vw_olap_observed_arrival AS
SELECT * FROM vw_olap_base
WHERE is_imputed = FALSE AND station_no > 1;

-- 5. Observed Origin Departure Delay View: Origin Stops Only (station_no = 1)
CREATE OR REPLACE VIEW vw_olap_observed_origin AS
SELECT * FROM vw_olap_base
WHERE is_imputed = FALSE AND station_no = 1;

-- 6. Prediction-Eligible View: Observed + Schedule-Aligned (is_imputed = FALSE AND has_schedule = TRUE)
CREATE OR REPLACE VIEW vw_olap_prediction_eligible AS
SELECT * FROM vw_olap_base
WHERE is_imputed = FALSE AND has_schedule = TRUE;

-- 7. Schedule-Missing View: Unaligned Stops (has_schedule = FALSE)
CREATE OR REPLACE VIEW vw_olap_schedule_missing AS
SELECT * FROM vw_olap_base
WHERE has_schedule = FALSE;

-- 8. Station Traffic Eligibility View (>= 1,000 Observed Stops/Year)
CREATE OR REPLACE VIEW vw_station_traffic_eligibility AS
SELECT 
    station_name,
    station_full_name,
    station_zone,
    COUNT(*) AS total_all_stops,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS observed_total_stops,
    COUNT(*) FILTER (WHERE is_imputed = FALSE AND station_no > 1) AS observed_arrival_stops,
    COUNT(*) FILTER (WHERE is_imputed = FALSE AND station_no = 1) AS observed_origin_stops,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS imputed_stops,
    COUNT(DISTINCT train_no) FILTER (WHERE is_imputed = FALSE) AS distinct_observed_trains,
    (COUNT(*) FILTER (WHERE is_imputed = FALSE) >= 1000) AS is_eligible_1000_stops,
    (COUNT(*) FILTER (WHERE is_imputed = FALSE AND station_no > 1) >= 1000) AS is_eligible_1000_arrival_stops
FROM vw_olap_base
GROUP BY station_name, station_full_name, station_zone;
""")

views_check_df = q("""
SELECT 
    (SELECT COUNT(*) FROM vw_olap_base) AS vw_olap_base_rows,
    (SELECT COUNT(*) FROM vw_olap_observed) AS vw_olap_observed_rows,
    (SELECT COUNT(*) FROM vw_olap_imputed) AS vw_olap_imputed_rows,
    (SELECT COUNT(*) FROM vw_olap_observed_arrival) AS vw_olap_observed_arrival_rows,
    (SELECT COUNT(*) FROM vw_olap_observed_origin) AS vw_olap_observed_origin_rows,
    (SELECT COUNT(*) FROM vw_olap_prediction_eligible) AS vw_olap_prediction_eligible_rows,
    (SELECT COUNT(*) FROM vw_olap_schedule_missing) AS vw_olap_schedule_missing_rows;
""")
print(views_check_df.to_string(index=False))
FINDINGS["step1_view_row_counts"] = views_check_df.iloc[0].to_dict()


=== STEP 1: CREATING REUSABLE ANALYTICAL VIEWS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 vw_olap_base_rows  vw_olap_observed_rows  vw_olap_imputed_rows  vw_olap_observed_arrival_rows  vw_olap_observed_origin_rows  vw_olap_prediction_eligible_rows  vw_olap_schedule_missing_rows
          36614824               35582318               1032506                       33773361                       1808957                          35279998                         858579


In [3]:
# ==========================================
# STEP 2: SEPARATE SCHEDULE-MISSING, OBSERVED-TARGET & IMPUTED-TARGET RECORDS
# ==========================================
print("\n=== STEP 2: 2x2 PARTITION MATRIX (is_imputed x has_schedule) ===")

partition_df = q("""
SELECT 
    CASE 
        WHEN is_imputed = FALSE AND has_schedule = TRUE THEN '1_observed_with_schedule (prediction_eligible)'
        WHEN is_imputed = FALSE AND has_schedule = FALSE THEN '2_observed_schedule_missing'
        WHEN is_imputed = TRUE AND has_schedule = TRUE THEN '3_imputed_with_schedule'
        WHEN is_imputed = TRUE AND has_schedule = FALSE THEN '4_imputed_schedule_missing'
    END AS partition_segment,
    is_imputed,
    has_schedule,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS pct_of_warehouse,
    COUNT(DISTINCT train_no) AS n_trains,
    COUNT(DISTINCT station_name) AS n_stations,
    ROUND(AVG(delay), 4) AS mean_delay_min,
    ROUND(MEDIAN(delay), 4) AS median_delay_min,
    ROUND(STDDEV(delay), 4) AS std_delay_min,
    ROUND(AVG(is_early_or_ontime) * 100.0, 4) AS early_or_ontime_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 4) AS on_time_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 4) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS high_risk_plus_critical_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_pct
FROM vw_olap_base
GROUP BY is_imputed, has_schedule
ORDER BY partition_segment;
""")
print(partition_df.to_string(index=False))

partition_csv = RESULT_DIR / "olap_step03_02_partition_summary.csv"
partition_df.to_csv(partition_csv, index=False)
print(f"\n-> Saved partition summary to {partition_csv.as_posix()}")

FINDINGS["step2_partition_matrix"] = partition_df.to_dict(orient="records")


=== STEP 2: 2x2 PARTITION MATRIX (is_imputed x has_schedule) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                             partition_segment  is_imputed  has_schedule   n_rows  pct_of_warehouse  n_trains  n_stations  mean_delay_min  median_delay_min  std_delay_min  early_or_ontime_0m_pct  on_time_15m_pct  delayed_gt_15m_pct  high_risk_plus_critical_pct  critical_pct
1_observed_with_schedule (prediction_eligible)       False          True 35279998           96.3544      6961        8143         35.6840              16.0        65.7689                 15.0254          49.3031             50.6969                       3.4644        1.7870
                   2_observed_schedule_missing       False         False   302320            0.8257      1881        1417         34.8785              12.0        74.6829                 22.4613          55.6109             44.3891                       4.2475        2.4322
                       3_imputed_with_schedule        True          True   476247            1.3007      2906        4363         34.5108              11.0        80.0685     

In [4]:
# ==========================================
# STEP 3: STANDARDIZED MEASURES ACROSS CORE ANALYTICAL DENOMINATORS
# ==========================================
print("\n=== STEP 3: STANDARDIZED MEASURES ACROSS DENOMINATORS ===")

denominators_df = q("""
WITH ViewUnion AS (
    SELECT '1_all_warehouse_records' AS view_name, * FROM vw_olap_base
    UNION ALL
    SELECT '2_observed_all_stops (default_olap)' AS view_name, * FROM vw_olap_observed
    UNION ALL
    SELECT '3_observed_arrival_only (station_no > 1)' AS view_name, * FROM vw_olap_observed_arrival
    UNION ALL
    SELECT '4_observed_origin_only (station_no = 1)' AS view_name, * FROM vw_olap_observed_origin
    UNION ALL
    SELECT '5_imputed_all_stops' AS view_name, * FROM vw_olap_imputed
    UNION ALL
    SELECT '6_prediction_eligible (obs + has_sched)' AS view_name, * FROM vw_olap_prediction_eligible
    UNION ALL
    SELECT '7_schedule_missing (has_sched = false)' AS view_name, * FROM vw_olap_schedule_missing
)
SELECT 
    view_name,
    COUNT(*) AS denominator_rows,
    ROUND(AVG(delay), 4) AS mean_delay_min,
    ROUND(MEDIAN(delay), 4) AS median_delay_min,
    ROUND(STDDEV(delay), 4) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 4) AS iqr_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.90), 4) AS p90_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.95), 4) AS p95_delay_min,
    ROUND(AVG(is_early) * 100.0, 4) AS early_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 4) AS exact_0m_pct,
    ROUND(AVG(is_early_or_ontime) * 100.0, 4) AS early_or_ontime_le_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 4) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 4) AS delayed_gt_15m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_to_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS total_bottleneck_gt_2sd_pct
FROM ViewUnion
GROUP BY view_name
ORDER BY view_name;
""")
print(denominators_df.to_string(index=False))

denom_csv = RESULT_DIR / "olap_step03_02_denominator_benchmarks.csv"
denominators_df.to_csv(denom_csv, index=False)
print(f"\n-> Saved denominator benchmarks to {denom_csv.as_posix()}")

FINDINGS["step3_denominator_benchmarks"] = denominators_df.to_dict(orient="records")


=== STEP 3: STANDARDIZED MEASURES ACROSS DENOMINATORS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                               view_name  denominator_rows  mean_delay_min  median_delay_min  std_delay_min  iqr_delay_min  p90_delay_min  p95_delay_min  early_lt_0m_pct  exact_0m_pct  early_or_ontime_le_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  high_risk_2sd_to_3sd_pct  critical_gt_3sd_pct  total_bottleneck_gt_2sd_pct
                 1_all_warehouse_records          36614824         35.5994              16.0        66.0268           36.0           88.0          137.0           1.6652       13.5539                    15.2191             49.5530             50.4470                    1.6756               1.8024                       3.4780
     2_observed_all_stops (default_olap)          35582318         35.6772              16.0        65.8498           36.0           88.0          137.0           1.6440       13.4446                    15.0886             49.3567             50.6433                    1.6786               1.7925                       3.4710
3_observed_arrival_

In [5]:
# ==========================================
# STEP 5: AUTOMATED PARTITION CONSERVATION CHECKS & EXPORT FINDINGS
# ==========================================
print("\n=== STEP 5: AUTOMATED CONSERVATION & DENOMINATOR CHECKS ===")

vc = FINDINGS["step1_view_row_counts"]

checks = {
    "obs_plus_imp_equals_total": bool(
        vc["vw_olap_observed_rows"] + vc["vw_olap_imputed_rows"] == vc["vw_olap_base_rows"]
    ),
    "obs_arrival_plus_obs_origin_equals_obs_total": bool(
        vc["vw_olap_observed_arrival_rows"] + vc["vw_olap_observed_origin_rows"] == vc["vw_olap_observed_rows"]
    ),
    "four_partitions_sum_to_total": bool(
        int(partition_df["n_rows"].sum()) == vc["vw_olap_base_rows"]
    )
}

for name, passed in checks.items():
    print(f"  {name}: {'PASSED' if passed else 'FAILED'}")

assert all(checks.values()), "Conservation check failed across analytical views!"
FINDINGS["step5_conservation_checks"] = checks

findings_path = RESULT_DIR / "step03_02_OLAP_FINDINGS.json"
with open(findings_path, "w") as f:
    json.dump(FINDINGS, f, indent=4)

print(f"\n-> Saved Step 3.2 findings to {findings_path.as_posix()}")
con.close()


=== STEP 5: AUTOMATED CONSERVATION & DENOMINATOR CHECKS ===
  obs_plus_imp_equals_total: PASSED
  obs_arrival_plus_obs_origin_equals_obs_total: PASSED
  four_partitions_sum_to_total: PASSED

-> Saved Step 3.2 findings to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step03_02_OLAP_FINDINGS.json
